# AI Agent Autopsy - 01 Tool Calling

This notebook is one of the companion notebooks for the talk **"AI Agent Autopsy: Frameworks, Platforms and Harnesses"**

It contains the demonstration code showing basic functionality of a set of frameworks.

In [ ]:
from anthropic import beta_tool
import pandas as pd
import anthropic

In [ ]:
import logging
import json

logging.basicConfig(filename='tool_calling.log', level=logging.INFO)

In [ ]:
client = anthropic.Anthropic()

<span style="color:red">Note:</span> Use of the anthropic models will require an API key that is defined either in a `.env` file or in your terminal session used to launch Jupyter. You can inspect yours with the following BASH command.

In [ ]:
%%bash 
echo $ANTHROPIC_API_KEY

## Example One - Basic Tool Calling

The anthropic framework allows you to easily define and use tools without needing to parse our requests and making the calls in your own control loop. This can all be handled using the function `beta_tool` --- Highly likely this naming might change.

In [ ]:
system_prompt = """
You are a data scientist using python to analyse a dataset that has been
loaded into Pandas dataframe with the following code
```
import pandas as pd
df = pd.read_csv(filename)
```

Make use of the available tools to generate a block of python code that will
answer the user's questions about the data.
"""

In [ ]:
df = None

@beta_tool
def get_column_names() -> list[str]:
    """Get the list of column names in the dataset being analysed.
    """
    global df
    col_names = list(df.columns)
    return json.dumps({"column_names": col_names})


@beta_tool
def get_column_types() -> list[str]:
    """Get the list of data types in the dataset being analysed.
    """
    col_types = [str(x) for x in list(df.dtypes)]
    return json.dumps({"column_types": col_types})


@beta_tool
def get_column_unique_value_counts() -> list[int]:
    """Get the list of data types in the dataset being analysed.
    """
    global df
    col_uniques = [len(df[x].unique()) for x in df.columns]
    return json.dumps({"column_uniques": col_uniques})

In [ ]:
def get_response(query):
    # Use the tool runner
    runner = client.beta.messages.tool_runner(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        tools=[get_column_names, get_column_types, get_column_unique_value_counts],
        system=system_prompt,
        messages=[
            {"role": "user", "content": query}
        ]
    )
    return message.content[0].text

In [ ]:
"""
This version will log the trace of tool calls so you can inspect what happens
"""
def get_response(query):
    # Use the tool runner
    runner = client.beta.messages.tool_runner(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        tools=[get_column_names, get_column_types, get_column_unique_value_counts],
        system=system_prompt,
        messages=[
            {"role": "user", "content": query}
        ]
    )
    logging.info("# RESPONSE LOG")
    for i, message in enumerate(runner):
        logging.info(f"[{i}] {message.id}")
        for part in message.content:
            logging.info(f"      {part}")
    return message.content[0].text

## Apply these functions to a query

In [ ]:
in_file = "data.csv"
df = pd.read_csv(in_file)

query = "Which field in the data has the most unique values?"
response = get_response(query)

print(response)